# 模块概述

WtBtCore 是 WonderTrader 回测框架的核心模块，负责提供各种策略类型的回测模拟环境。主要包括：
- 历史数据回放和管理
- 多种策略类型的回测模拟器（CTA、选股、高频、UFT、执行器）
- 订单撮合引擎
- 事件通知机制
- 辅助工具类

1. **数据层**（HisDataMgr + HisDataReplayer）：
   - 负责历史数据的加载、缓存和回放
   - 支持多种数据类型和回放模式
   - 是整个框架的数据基础

2. **策略层**（各种Mocker）：
   - 提供不同策略类型的回测模拟环境
   - 每个Mocker都实现相应的策略上下文接口和数据接收接口
   - 负责策略执行、持仓管理、盈亏计算等

3. **辅助层**（MatchEngine + EventNotifier）：
   - MatchEngine提供订单撮合功能
   - EventNotifier提供事件通知功能

4. **工具层**（WtHelper）：
   - 提供通用的辅助功能
   - 所有类都可以使用


# 层次关系图
```mermaid
graph TB
    %% 数据层
    subgraph 数据层["数据层 - 历史数据管理"]
        HisDataMgr["HisDataMgr<br/>历史数据管理器<br/>- 动态加载数据读取器模块<br/>- 提供统一的数据加载接口<br/>- 支持K线/Tick/订单队列等数据"]
        HisDataReplayer["HisDataReplayer<br/>历史数据回放器<br/>- 数据缓存管理<br/>- 按K线/Tick/定时任务回放<br/>- 数据订阅和查询<br/>- 复权处理"]
        
        HisDataMgr -->|"使用"| HisDataReplayer
    end
    
    %% 策略模拟器层
    subgraph 策略层["策略层 - 回测模拟器"]
        CtaMocker["CtaMocker<br/>CTA策略回测模拟器<br/>- 继承ICtaStraCtx和IDataSink<br/>- 支持条件单、限价单、止损单<br/>- 持仓管理和盈亏计算<br/>- 图表数据输出"]
        
        SelMocker["SelMocker<br/>选股策略回测模拟器<br/>- 继承ISelStraCtx和IDataSink<br/>- 信号延迟执行机制<br/>- 多明细持仓管理<br/>- T+1规则支持"]
        
        HftMocker["HftMocker<br/>高频交易策略回测模拟器<br/>- 继承IHftStraCtx和IDataSink<br/>- 支持Tick/订单队列/订单明细/逐笔成交<br/>- 订单队列机制<br/>- 异步回测模式"]
        
        UftMocker["UftMocker<br/>UFT极速策略回测模拟器<br/>- 继承IUftStraCtx和IDataSink<br/>- 订单队列机制<br/>- 多空双向持仓<br/>- T+1规则支持"]
        
        ExecMocker["ExecMocker<br/>执行器模拟器<br/>- 继承ExecuteContext/IDataSink/IMatchSink<br/>- 通过撮合引擎执行订单<br/>- 支持多种数量模式<br/>- 订单执行日志"]
    end
    
    %% 辅助组件层
    subgraph 辅助层["辅助层 - 核心组件"]
        MatchEngine["MatchEngine<br/>撮合引擎<br/>- 订单管理<br/>- 限价订单簿维护<br/>- 订单撮合逻辑<br/>- 撤单处理"]
        
        EventNotifier["EventNotifier<br/>事件通知器<br/>- 消息队列服务<br/>- 回测事件通知<br/>- 数据推送<br/>- 资金信息推送"]
    end
    
    %% 工具层
    subgraph 工具层["工具层 - 通用工具"]
        WtHelper["WtHelper<br/>辅助工具类<br/>- 路径管理<br/>- 目录创建<br/>- 跨平台支持<br/>- 静态工具方法"]
    end
    
    %% 接口层（虚拟）
    subgraph 接口层["接口层 - 抽象接口"]
        IDataSink["IDataSink<br/>数据接收接口<br/>- handle_tick<br/>- handle_bar_close<br/>- handle_schedule<br/>- handle_init等"]
        
        ICtaStraCtx["ICtaStraCtx<br/>CTA策略上下文接口"]
        ISelStraCtx["ISelStraCtx<br/>选股策略上下文接口"]
        IHftStraCtx["IHftStraCtx<br/>高频策略上下文接口"]
        IUftStraCtx["IUftStraCtx<br/>UFT策略上下文接口"]
        ExecuteContext["ExecuteContext<br/>执行器上下文接口"]
        IMatchSink["IMatchSink<br/>撮合回调接口"]
    end
    
    %% 数据层关系
    HisDataReplayer -->|"实现"| IDataSink
    
    %% 策略层关系
    CtaMocker -->|"实现"| ICtaStraCtx
    CtaMocker -->|"实现"| IDataSink
    CtaMocker -->|"使用"| HisDataReplayer
    CtaMocker -.->|"可选使用"| EventNotifier
    
    SelMocker -->|"实现"| ISelStraCtx
    SelMocker -->|"实现"| IDataSink
    SelMocker -->|"使用"| HisDataReplayer
    
    HftMocker -->|"实现"| IHftStraCtx
    HftMocker -->|"实现"| IDataSink
    HftMocker -->|"使用"| HisDataReplayer
    
    UftMocker -->|"实现"| IUftStraCtx
    UftMocker -->|"实现"| IDataSink
    UftMocker -->|"使用"| HisDataReplayer
    
    ExecMocker -->|"实现"| ExecuteContext
    ExecMocker -->|"实现"| IDataSink
    ExecMocker -->|"实现"| IMatchSink
    ExecMocker -->|"使用"| HisDataReplayer
    ExecMocker -->|"使用"| MatchEngine
    
    %% 辅助层关系
    MatchEngine -->|"回调"| IMatchSink
    HisDataReplayer -.->|"可选使用"| EventNotifier
    
    %% 工具层关系（所有类都使用）
    CtaMocker -.->|"使用"| WtHelper
    SelMocker -.->|"使用"| WtHelper
    HftMocker -.->|"使用"| WtHelper
    UftMocker -.->|"使用"| WtHelper
    ExecMocker -.->|"使用"| WtHelper
    HisDataReplayer -.->|"使用"| WtHelper
    HisDataMgr -.->|"使用"| WtHelper
    EventNotifier -.->|"使用"| WtHelper
    MatchEngine -.->|"使用"| WtHelper
    
    %% 样式定义
    classDef dataLayer fill:#e1f5ff,stroke:#01579b,stroke-width:2px
    classDef strategyLayer fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef helperLayer fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef toolLayer fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px
    classDef interfaceLayer fill:#fce4ec,stroke:#880e4f,stroke-width:2px,stroke-dasharray: 5 5
    
    class HisDataMgr,HisDataReplayer dataLayer
    class CtaMocker,SelMocker,HftMocker,UftMocker,ExecMocker strategyLayer
    class MatchEngine,EventNotifier helperLayer
    class WtHelper toolLayer
    class IDataSink,ICtaStraCtx,ISelStraCtx,IHftStraCtx,IUftStraCtx,ExecuteContext,IMatchSink interfaceLayer
```


# 数据层

## HisDataMgr.h/cpp - 历史数据管理器
管理历史数据的加载和读取，通过动态库加载数据读取器模块

```cpp
class HisDataMgr : public IBtDtReaderSink
```

参考 [Includes/note.ipynb/数据管理接口层/回测数据读取 IBtDtReader.h/回测数据读取回调接口类 IBtDtReaderSink](../Includes/note.ipynb)

### 成员
- `IBtDtReader* _reader`：数据读取器指针

### IBtDtReaderSink 接口实现

#### 数据读取器日志回调 reader_log

### 初始化 init

### 数据加载

#### 加载原始K线数据 load_raw_bars

#### 加载原始Tick数据 load_raw_ticks

#### 加载原始订单队列数据 load_raw_ordque

#### 加载原始订单明细数据 load_raw_orddtl

#### 加载原始逐笔成交数据 load_raw_trans

## HisDataReplayer.h/cpp - 历史数据回放器
历史数据的回放和模拟，是整个回测框架的数据核心

```mermaid
graph TB
    %% 接口层
    IDataSink["IDataSink<br/>数据接收器接口<br/>接收回放数据"]
    IBtDataLoader["IBtDataLoader<br/>数据加载器接口<br/>可选外部数据源"]
    
    %% 主类
    HisDataReplayer["HisDataReplayer<br/>历史数据回放器<br/>数据加载/缓存/回放核心"]
    
    %% 外部依赖
    HisDataMgr["HisDataMgr<br/>数据管理器"]
    
    %% 策略模拟器
    Mocker["策略模拟器<br/>CtaMocker/SelMocker等<br/>实现IDataSink"]
    
    %% 核心关系
    HisDataReplayer -->|"注册回调"| IDataSink
    HisDataReplayer -.->|"可选"| IBtDataLoader
    HisDataReplayer -->|"使用"| HisDataMgr
    HisDataReplayer -.->|"推送数据"| Mocker
    
    Mocker -.->|"实现"| IDataSink
    Mocker -->|"使用"| HisDataReplayer
    
    %% 样式
    classDef interface fill:#fce4ec,stroke:#880e4f,stroke-width:2px,stroke-dasharray:5 5
    classDef main fill:#e1f5ff,stroke:#01579b,stroke-width:3px
    classDef internal fill:#fff3e0,stroke:#e65100,stroke-width:2px
    classDef cache fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    classDef external fill:#e0f2f1,stroke:#004d40,stroke-width:2px
    classDef mocker fill:#f1f8e9,stroke:#33691e,stroke-width:2px
    
    class IDataSink,IBtDataLoader interface
    class HisDataReplayer main
    class TickCache,BarsCache cache
    class Mocker mocker
```

### IDataSink

### IBtDataLoader

### HisDataReplayer

# 策略层

## CtaMocker.h/cpp - CTA策略回测模拟器
**作用**：模拟CTA（Commodity Trading Advisor）策略在历史数据上的交易执行过程

**核心特性**：
1. **策略执行环境**：
   - 实现ICtaStraCtx接口，为策略提供交易上下文
   - 实现IDataSink接口，接收历史数据回放器推送的数据

2. **订单类型支持**：
   - 条件单：价格满足条件时自动执行
   - 限价单：指定价格下单
   - 止损单：价格触发止损时执行

3. **持仓管理**：
   - 多空双向持仓
   - 持仓明细追踪（每笔开仓的详细信息）
   - T+1规则支持（冻结持仓）

4. **盈亏计算**：
   - 持仓动态盈亏
   - 已平仓盈亏
   - 最大盈利/最大亏损追踪

5. **数据输出**：
   - 成交记录CSV
   - 平仓记录CSV
   - 资金曲线CSV
   - 信号记录CSV
   - 持仓记录CSV
   - 策略状态JSON
   - 图表数据JSON/CSV（K线、指标、交易标记）

6. **增量回测**：
   - 支持从上次回测结果继续回测

7. **异步回测**：
   - 通过钩子机制实现步进式回测

**依赖关系**：
- 依赖：`HisDataReplayer`（数据源）、`EventNotifier`（可选，事件通知）
- 被依赖：外部回测程序使用

**核心接口**：
- `init_cta_factory()`：初始化CTA策略工厂
- `stra_enter_long()`：开多仓
- `stra_enter_short()`：开空仓
- `stra_exit_long()`：平多仓
- `stra_exit_short()`：平空仓
- `stra_set_position()`：设置目标仓位
- `stra_get_position()`：获取持仓数量

**用法**：
```cpp
CtaMocker mocker(replayer, "strategy_name", slippage, persistData, notifier);
mocker.init_cta_factory(cfg);
// 策略通过ICtaStraCtx接口进行交易操作
```

## SelMocker.h/cpp - 选股策略回测模拟器
**作用**：模拟选股策略在历史数据上的执行过程

**核心特性**：
1. **信号延迟执行机制**：
   - 策略发出的持仓信号会在下一个tick执行
   - 模拟真实交易延迟

2. **多明细持仓管理**：
   - 每笔持仓都有独立的明细记录
   - 支持精确的盈亏计算

3. **T+1规则支持**：
   - 正确处理T+1市场的冻结持仓
   - 交易日开始时释放冻结持仓

4. **滑点模拟**：
   - 支持绝对滑点（单位：最小变动价位）
   - 支持比例滑点（单位：万分之一）

5. **数据输出**：
   - 交易日志、平仓日志、资金日志
   - 信号日志、持仓日志
   - 策略状态JSON

**依赖关系**：
- 依赖：`HisDataReplayer`（数据源）
- 被依赖：外部回测程序使用

**核心接口**：
- `init_sel_factory()`：初始化选股策略工厂
- `stra_set_position()`：设置目标持仓
- `stra_get_position()`：获取持仓数量

**与CtaMocker的区别**：
- SelMocker使用信号机制，信号延迟到下一个tick执行
- CtaMocker支持条件单、限价单、止损单等复杂订单类型
- SelMocker专注于选股场景，CtaMocker专注于CTA交易场景

## HftMocker.h/cpp - 高频交易策略回测模拟器
**作用**：模拟高频交易（HFT）策略在历史数据上的执行过程

**核心特性**：
1. **高频数据支持**：
   - Tick数据
   - 订单队列数据
   - 订单明细数据
   - 逐笔成交数据

2. **订单队列机制**：
   - 策略发出的订单进入订单队列
   - 在tick数据到来时进行撮合

3. **异步回测模式**：
   - 通过钩子机制实现步进式回测
   - 支持单步tick处理

4. **持仓管理**：
   - 多空双向持仓
   - 持仓明细追踪

**依赖关系**：
- 依赖：`HisDataReplayer`（数据源）
- 被依赖：外部回测程序使用

**核心接口**：
- `init_hft_factory()`：初始化HFT策略工厂
- `stra_buy()`：买入订单
- `stra_sell()`：卖出订单
- `stra_cancel()`：撤单
- `install_hook()`：安装钩子（异步回测）
- `step_tick()`：步进tick（异步回测）

**与UftMocker的区别**：
- HftMocker和UftMocker都支持高频数据
- UftMocker更专注于极速交易场景
- 两者在订单处理机制上略有不同

## UftMocker.h/cpp - UFT极速策略回测模拟器
**作用**：模拟UFT（Ultra Fast Trading）极速策略在历史数据上的执行过程

**核心特性**：
1. **订单队列机制**：
   - 策略发出的订单进入订单队列
   - 在tick数据到来时进行撮合

2. **多空双向持仓**：
   - 支持同时持有多头和空头持仓
   - 区分昨仓和今仓

3. **T+1规则支持**：
   - 正确处理T+1市场的冻结持仓
   - 交易日开始时进行持仓转换

4. **订单撮合**：
   - 支持使用最新价或对手价撮合
   - 支持错误率模拟（订单被随机撤销）

**依赖关系**：
- 依赖：`HisDataReplayer`（数据源）
- 被依赖：外部回测程序使用

**核心接口**：
- `init_uft_factory()`：初始化UFT策略工厂
- `stra_enter_long()`：开多
- `stra_enter_short()`：开空
- `stra_exit_long()`：平多
- `stra_exit_short()`：平空
- `stra_buy()`：智能买入（先平空再开多）
- `stra_sell()`：智能卖出（先平多再开空）

## ExecMocker.h/cpp - 执行器模拟器
**作用**：模拟执行器在历史数据上的订单执行过程

**核心特性**：
1. **撮合引擎集成**：
   - 通过MatchEngine进行订单撮合
   - 模拟真实的订单执行过程

2. **多种数量模式**：
   - 反复正负模式
   - 一直买入模式
   - 一直卖出模式

3. **订单执行日志**：
   - 记录信号时间、下单时间、成交时间
   - 记录订单执行统计信息

**依赖关系**：
- 依赖：`HisDataReplayer`（数据源）、`MatchEngine`（撮合引擎）
- 被依赖：外部回测程序使用

**核心接口**：
- `init()`：初始化执行器模拟器
- `buy()`：买入订单
- `sell()`：卖出订单
- `cancel()`：撤单

**与其他Mocker的区别**：
- ExecMocker专注于订单执行，不涉及策略逻辑
- 通过撮合引擎模拟真实订单执行过程
- 主要用于测试执行器策略的性能

# 辅助层

## MatchEngine.h/cpp - 撮合引擎
**作用**：模拟订单撮合过程，包括订单管理、撮合逻辑、限价订单簿维护

**核心功能**：
1. **订单管理**：
   - 接收买入/卖出订单
   - 管理订单状态（待激活、已激活、待撤单、已撤单）
   - 订单排队机制

2. **限价订单簿（LOB）维护**：
   - 维护价格档位的订单队列信息
   - 更新买一价、卖一价
   - 模拟订单在价格档位上的排队位置

3. **订单撮合**：
   - 根据tick数据检查订单是否可成交
   - 支持限价单和市价单的撮合
   - 支持主动订单（对手价）和被动订单（挂单）

4. **撤单处理**：
   - 支持按订单ID撤单
   - 支持按合约和方向撤单
   - 支持撤单率模拟

**依赖关系**：
- 依赖：`WTSTickData`（Tick数据）、`IMatchSink`（回调接口）
- 被依赖：`ExecMocker` 使用

**核心接口**：
- `init()`：初始化撮合引擎
- `regisSink()`：注册回调接口
- `handle_tick()`：处理tick数据
- `buy()`：买入订单
- `sell()`：卖出订单
- `cancel()`：撤单

**用法**：
```cpp
MatchEngine engine;
engine.init(cfg);
engine.regisSink(mocker);
engine.handle_tick(stdCode, tick);
OrderIDs ids = engine.buy(stdCode, price, qty, curTime);
```

## EventNotifier.h/cpp - 事件通知器
**作用**：在回测过程中向外发送事件和数据通知

**核心功能**：
1. **消息队列服务**：
   - 通过动态库加载消息队列服务模块
   - 建立通信通道

2. **事件通知**：
   - notifyEvent：发送回测事件通知（如回测开始、结束等）
   - notifyData：推送原始数据
   - notifyFund：推送资金信息（JSON格式）

**依赖关系**：
- 依赖：`WtHelper`（路径管理）
- 被依赖：`CtaMocker`（可选）、`HisDataReplayer`（可选）

**核心接口**：
- `init()`：初始化事件通知器
- `notifyEvent()`：通知事件
- `notifyData()`：通知数据
- `notifyFund()`：通知资金信息

**用法**：
```cpp
EventNotifier notifier;
notifier.init(cfg);
notifier.notifyEvent("BACKTEST_START");
notifier.notifyFund("FUND_INFO", date, profit, dynprofit, balance, fee);
```

# 工具层

## WtHelper.h/cpp
**作用**：提供WonderTrader框架中的通用辅助功能

**核心功能**：
1. **路径管理**：
   - getCWD()：获取当前工作目录
   - getOutputDir()：获取输出目录（如果不存在会自动创建）
   - setOutputDir()：设置输出目录

2. **实例目录管理**：
   - getInstDir()：获取实例目录
   - setInstDir()：设置实例目录

3. **跨平台支持**：
   - 支持Windows和Unix平台的路径操作
   - 统一路径分隔符格式

**特点**：
- 所有方法都是静态方法，无需实例化即可使用
- 使用静态变量缓存结果，提高性能

**依赖关系**：
- 无依赖（最底层工具类）
- 被依赖：所有其他类都使用

**用法**：
```cpp
std::string cwd = WtHelper::getCWD();
const char* outDir = WtHelper::getOutputDir();
WtHelper::setOutputDir("./custom_output/");
```
